# 05 · Decoding: greedy, temperature, top-k, top-p, penalties and beam search

After this notebook you can turn logits into text in every standard way - greedy, temperature, top-k, top-p (nucleus), min-p, repetition/frequency/presence penalties and beam search - implemented from scratch, see what each does to the distribution, read token logprobs from the API, and pick sensible settings for extraction, code and creative work.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [04 · Tiny GPT](04_transformer_block_and_tiny_gpt.ipynb), [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb)

## Why this matters in interviews

- "What is temperature doing, mathematically?" and "top-k or top-p?" are asked in almost every LLM interview; the answer is three lines of NumPy.
- Production bugs trace back to decoding: repetition loops, "temperature 0 but outputs still differ", JSON broken by high temperature, a reasoning model rejecting `temperature`.
- Logprobs are a cheap, underused signal: classification confidence, abstention, detecting hallucination-prone answers.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo22` | What are logits, and what is the softmax doing? |
| `fo14` | What is temperature actually doing, mathematically? |
| `fo15` | What are top-k and top-p, and which should you use? |
| `fo16` | What are max tokens, stop sequences and the repetition penalty? |
| `fo07` | Why does the same prompt give a different answer each time? |
| `tr34` | Why does the model sometimes repeat itself in a loop? |
| `fo23` | What are logprobs and what can you legitimately use them for? |
| `in32` | How do you make LLM output reproducible for testing? |

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

rng = np.random.default_rng(0)
pd.set_option("display.width", 120)
print("ready")

## 1. Logits -> probabilities

**In plain English:** at every step the model outputs one raw score (a *logit*) per vocabulary token. Softmax turns the scores into probabilities: exponentiate (so everything is positive and bigger scores win disproportionately), then divide by the total (so they sum to 1). Decoding is the policy for picking a token from that distribution.

A toy next-token distribution for the prompt *"The cat sat on the"*:

In [ ]:
VOCAB = ["mat", "sofa", "floor", "bed", "roof", "chair", "table", "grass", "keyboard", "moon", "banana", "quantum"]
LOGITS = np.array([4.0, 3.2, 3.0, 2.6, 2.0, 1.9, 1.6, 1.2, 0.8, -0.5, -1.5, -3.0])

def softmax(z):
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

probs = softmax(LOGITS)
print(pd.DataFrame({"token": VOCAB, "logit": LOGITS, "prob": probs.round(4)}).to_string(index=False))
assert np.isclose(probs.sum(), 1.0)
assert np.allclose(softmax(LOGITS + 100), probs), "adding a constant to every logit changes nothing"
print(f"\ngreedy (argmax) picks: {VOCAB[int(np.argmax(probs))]!r} with p = {probs.max():.2f}")

Only differences between logits matter (the shift-invariance assert). **Greedy decoding** always takes the argmax: deterministic, good for short factual answers, but prone to bland and repetitive text over long outputs (section 5).

## 2. Temperature

Divide the logits by T before the softmax: `p = softmax(z / T)`.

- **T < 1** sharpens the distribution (differences between logits are amplified); as T -> 0 it becomes greedy.
- **T = 1** is the model's own distribution.
- **T > 1** flattens it towards uniform: more surprising tokens, more mistakes.

In [ ]:
def entropy(p):
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

temps = [0.2, 0.7, 1.0, 1.5]
fig, axes = plt.subplots(1, 4, figsize=(11, 3.2), sharey=True)
for ax, t in zip(axes, temps):
    p_t = softmax(LOGITS / t)
    ax.bar(range(len(VOCAB)), p_t, color=plt.cm.viridis(t / 1.6))
    ax.set_title(f"T = {t}  (entropy {entropy(p_t):.2f} bits)", fontsize=9)
    ax.set_xticks(range(len(VOCAB)), VOCAB, rotation=90, fontsize=7)
axes[0].set_ylabel("probability")
fig.suptitle("Temperature reshapes the same logits: sharp at low T, flat at high T", fontsize=10)
plt.tight_layout(); plt.show()

ents = [entropy(softmax(LOGITS / t)) for t in temps]
assert ents == sorted(ents), "entropy grows with temperature"
assert np.allclose(softmax(LOGITS / 1.0), probs)
assert softmax(LOGITS / 0.01).max() > 0.999, "T -> 0 is greedy"
for t in temps:
    p_t = softmax(LOGITS / t)
    print(f"T={t}: p('mat')={p_t[0]:.2f}  p('banana')={p_t[VOCAB.index('banana')]:.5f}")

Temperature never changes the *order* of tokens, only how peaked the distribution is. At T=1.5 "banana" becomes plausible enough to appear now and then in a long generation - that is where nonsense comes from. Note: `temperature=0` in an API means "greedy" (dividing by zero is special-cased).

## 3. Truncation: top-k, top-p (nucleus) and min-p

Instead of reshaping, *cut off* the long tail of unlikely tokens, then renormalise and sample from what is left.

- **top-k:** keep the k most likely tokens. Simple, but k is fixed: too many when the model is sure, too few when many continuations are fine.
- **top-p (nucleus):** keep the smallest set of top tokens whose total probability reaches p. The set *adapts*: small when the model is confident, large when it is not.
- **min-p:** keep tokens with probability at least `min_p x p_max`. Also adaptive; popular in open-source samplers (llama.cpp, vLLM).

In [ ]:
def top_k_filter(p, k):
    keep = np.zeros_like(p, dtype=bool)
    keep[np.argsort(-p)[:k]] = True
    q = np.where(keep, p, 0.0)
    return q / q.sum()

def top_p_filter(p, top_p):
    order = np.argsort(-p)
    cum = np.cumsum(p[order])
    n_keep = int(np.searchsorted(cum, top_p) + 1)          # smallest prefix with mass >= top_p
    keep = np.zeros_like(p, dtype=bool)
    keep[order[:n_keep]] = True
    q = np.where(keep, p, 0.0)
    return q / q.sum()

def min_p_filter(p, min_p):
    q = np.where(p >= min_p * p.max(), p, 0.0)
    return q / q.sum()

for name, q in [("top-k (k=3)", top_k_filter(probs, 3)), ("top-p (p=0.9)", top_p_filter(probs, 0.9)),
                ("min-p (0.1)", min_p_filter(probs, 0.1))]:
    kept = [VOCAB[i] for i in np.flatnonzero(q)]
    print(f"{name:14s} keeps {len(kept):2d}: {kept}")
    assert np.isclose(q.sum(), 1.0)

In [ ]:
order = np.argsort(-probs)
cum = np.cumsum(probs[order])
n_keep = int(np.count_nonzero(top_p_filter(probs, 0.9)))
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(range(len(VOCAB)), probs[order], color=["tab:green" if i < n_keep else "lightgray" for i in range(len(VOCAB))])
ax2 = ax.twinx()
ax2.plot(range(len(VOCAB)), cum, "k.-", label="cumulative mass")
ax2.axhline(0.9, color="tab:red", ls="--", label="top_p = 0.9")
ax2.set_ylim(0, 1.05); ax2.set_ylabel("cumulative probability"); ax2.legend(loc="center right")
ax.set_xticks(range(len(VOCAB)), [VOCAB[i] for i in order], rotation=45, fontsize=8)
ax.set_ylabel("probability")
ax.set_title(f"Nucleus sampling keeps the green tokens: {n_keep} tokens hold {cum[n_keep - 1]:.0%} of the mass")
plt.tight_layout(); plt.show()

The nucleus adapts to the model's confidence. Compare a confident distribution (the answer to *"The capital of France is"*) with an open-ended one (*"My favourite colour is"*):

In [ ]:
confident = softmax(np.array([9.0, 4.0, 3.5, 3.0, 2.0, 1.0, 0.5, 0.0, -1.0, -2.0]))
open_ended = softmax(np.array([2.0, 1.9, 1.8, 1.8, 1.7, 1.6, 1.5, 1.2, 1.0, 0.5]))
for name, p in [("confident", confident), ("open-ended", open_ended)]:
    print(f"{name:11s}: top-p(0.9) keeps {np.count_nonzero(top_p_filter(p, 0.9))} tokens, "
          f"top-k(3) keeps 3 tokens holding {np.sort(p)[::-1][:3].sum():.0%} of the mass")
assert np.count_nonzero(top_p_filter(confident, 0.9)) < np.count_nonzero(top_p_filter(open_ended, 0.9))

With k=3, the open-ended case throws away most of the plausible mass while the confident case keeps two junk tokens; top-p keeps 1 token in the first case and most tokens in the second. That is the argument for top-p (or min-p) over top-k. In practice: tune **either** temperature **or** top_p, not both at once (OpenAI's own advice), and leave top_p at 1 if you use temperature.

### Putting it together: one sampler

Servers apply the steps in a fixed order - penalties on logits, then temperature, then truncation, then a random draw:

```
logits -> penalties -> / temperature -> softmax -> top-k -> top-p / min-p -> renormalise -> sample
```

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, min_p=None, rng=rng):
    if temperature == 0:
        return int(np.argmax(logits))                      # greedy
    p = softmax(np.asarray(logits, dtype=float) / temperature)
    if top_k:
        p = top_k_filter(p, top_k)
    if top_p is not None and top_p < 1:
        p = top_p_filter(p, top_p)
    if min_p:
        p = min_p_filter(p, min_p)
    return int(rng.choice(len(p), p=p))

draws = Counter(VOCAB[sample_next(LOGITS, temperature=0.7, top_p=0.9)] for _ in range(5000))
expected = top_p_filter(softmax(LOGITS / 0.7), 0.9)
print("empirical vs expected frequency (T=0.7, top_p=0.9):")
for i in np.flatnonzero(expected):
    print(f"  {VOCAB[i]:6s} {draws[VOCAB[i]] / 5000:.3f} vs {expected[i]:.3f}")
assert all(abs(draws[VOCAB[i]] / 5000 - expected[i]) < 0.02 for i in range(len(VOCAB)))
assert "banana" not in draws, "truncated tokens can never be sampled"

## 4. A toy language model to decode with

To see repetition, penalties and diversity we need a model that generates *sequences*. A word-level bigram model (counts of which word follows which) trained on a few sentences is enough: `P(next | current)`.

In [ ]:
CORPUS = """the cat sat on the mat . the dog sat on the rug . the cat saw the dog and the dog saw the cat .
the cat ran to the door and the dog ran after the cat . a bird sat on the roof and sang a song .
the dog slept on the mat . the cat chased a bird in the garden . the bird flew over the garden wall .
a dog barked at the door and the cat hid under the bed . the sun was warm and the garden was quiet ."""
words = CORPUS.split()
LM_VOCAB = sorted(set(words))
w2i = {w: i for i, w in enumerate(LM_VOCAB)}
counts = np.full((len(LM_VOCAB), len(LM_VOCAB)), 0.01)            # add-0.01 smoothing
for a, b in zip(words, words[1:]):
    counts[w2i[a], w2i[b]] += 1
LM_LOGITS = np.log(counts / counts.sum(axis=1, keepdims=True))     # log P(next | current) as "logits"
print(f"vocab {len(LM_VOCAB)} words; after 'the', top next words:",
      [(LM_VOCAB[j], round(float(np.exp(LM_LOGITS[w2i['the'], j])), 2)) for j in np.argsort(-LM_LOGITS[w2i['the']])[:4]])

def generate_lm(start, n=20, temperature=0.0, top_p=None, freq_pen=0.0, pres_pen=0.0, rng=None):
    rng = rng or np.random.default_rng(0)
    seq = [w2i[start]]
    for _ in range(n):
        logits = LM_LOGITS[seq[-1]].copy()
        c = np.bincount(seq, minlength=len(LM_VOCAB))
        logits -= freq_pen * c + pres_pen * (c > 0)                 # OpenAI-style penalties
        seq.append(sample_next(logits, temperature=temperature, top_p=top_p, rng=rng))
    return [LM_VOCAB[i] for i in seq]

## 5. Repetition loops and the penalties that fix them

Greedy decoding with a model that has a favourite continuation falls into a loop: once it is in a state it has been in before, it makes the same choice again, forever.

- **Frequency penalty** (OpenAI, -2..2): `logit[j] -= freq_pen x count(j)` - the more a token has already appeared, the bigger the penalty.
- **Presence penalty** (OpenAI, -2..2): `logit[j] -= pres_pen x [count(j) > 0]` - a flat penalty for having appeared at all (pushes towards new topics).
- **Repetition penalty** (Hugging Face / llama.cpp, >1): divides positive logits (multiplies negative ones) of seen tokens by the penalty.

In [ ]:
def distinct_ratio(seq):
    return len(set(seq)) / len(seq)

greedy = generate_lm("the", n=24)
penal = generate_lm("the", n=24, freq_pen=0.8, pres_pen=0.3)
print("greedy          :", " ".join(greedy))
print("greedy + penalty:", " ".join(penal))
print(f"distinct words: {distinct_ratio(greedy):.2f} -> {distinct_ratio(penal):.2f}")
assert distinct_ratio(penal) > distinct_ratio(greedy)

def hf_repetition_penalty(logits, seen_ids, penalty=1.3):
    out = logits.copy()
    s = np.array(sorted(set(seen_ids)))
    out[s] = np.where(out[s] > 0, out[s] / penalty, out[s] * penalty)
    return out

demo = hf_repetition_penalty(np.array([2.0, -1.0, 0.5]), [0, 1], penalty=2.0)
print("HF repetition_penalty=2 on logits [2.0, -1.0, 0.5] with tokens 0,1 seen ->", demo)
assert np.allclose(demo, [1.0, -2.0, 0.5])

Large language models loop for similar reasons (tr34): greedy or low-temperature decoding, a strong pattern in the context that attention keeps copying, and a model that has seen repetitive text. Fixes, in order: a little temperature, a mild frequency/presence penalty (0.1-0.5), better prompts, and a `max_tokens` cap. Too much penalty breaks things that *should* repeat - variable names in code, JSON keys, a person's name.

## 6. Beam search

Greedy picks the best *next* token; **beam search** keeps the `B` best *partial sequences* at every step and returns the most probable full sequence. The classic example where greedy is wrong:

In [ ]:
TREE = {  # P(next | sequence so far) for a toy model
    ("The",): {"nice": 0.5, "dog": 0.4, "car": 0.1},
    ("The", "nice"): {"woman": 0.4, "house": 0.3, "guy": 0.3},
    ("The", "dog"): {"has": 0.9, "runs": 0.05, "and": 0.05},
    ("The", "car"): {"is": 0.3, "drives": 0.5, "turns": 0.2},
}

def greedy_tree(start=("The",), steps=2):
    seq, lp = start, 0.0
    for _ in range(steps):
        tok, p = max(TREE[seq].items(), key=lambda kv: kv[1])
        seq, lp = seq + (tok,), lp + math.log(p)
    return seq, lp

def beam_search(start=("The",), steps=2, beam=2):
    beams = [(start, 0.0)]
    for _ in range(steps):
        cands = [(seq + (tok,), lp + math.log(p)) for seq, lp in beams for tok, p in TREE[seq].items()]
        beams = sorted(cands, key=lambda c: -c[1])[:beam]            # keep the best `beam` sequences
    return beams[0]

g_seq, g_lp = greedy_tree()
b_seq, b_lp = beam_search(beam=2)
print(f"greedy     : {' '.join(g_seq):16s} P = {math.exp(g_lp):.2f}")
print(f"beam (B=2) : {' '.join(b_seq):16s} P = {math.exp(b_lp):.2f}")
assert b_lp > g_lp, "beam search found a more probable sequence than greedy"

Greedy took "nice" (0.5) and ended with P=0.20; beam search kept "dog" alive and found "The dog has" with P=0.36. Beam search is the standard for tasks with one correct output - **translation, speech recognition (Whisper), summarisation** in encoder-decoder models. For open-ended chat it produces bland, repetitive text (the most probable text is rarely the most human-like), so chat APIs sample instead and do not even expose beams. Length normalisation (divide log-prob by length^alpha) stops beams from preferring short outputs.

## 7. Diversity vs quality: sampling many times

Generate 300 continuations at each temperature and measure **diversity** (distinct word pairs across all samples) and **quality** (average log-probability per word under the model at T=1 - a stand-in for "how plausible is this text").

In [ ]:
def sample_stats(temperature, top_p=None, n_samples=300, length=10):
    r = np.random.default_rng(42)
    bigrams, lps = set(), []
    for _ in range(n_samples):
        seq = generate_lm("the", n=length, temperature=temperature, top_p=top_p, rng=r)
        ids = [w2i[w] for w in seq]
        bigrams.update(zip(seq, seq[1:]))
        lps.append(np.mean([LM_LOGITS[a, b] for a, b in zip(ids, ids[1:])]))
    return len(bigrams), float(np.mean(lps))

temps_q = [0.3, 0.6, 1.0, 1.3, 1.8]
rows = [(t, *sample_stats(t)) for t in temps_q]
rows_p = [(t, *sample_stats(t, top_p=0.9)) for t in temps_q]
stats = pd.DataFrame(rows, columns=["T", "distinct bigrams", "avg log-prob"])
stats["distinct (top_p=0.9)"] = [r[1] for r in rows_p]
stats["avg log-prob (top_p=0.9)"] = [r[2] for r in rows_p]
print(stats.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(stats["distinct bigrams"], stats["avg log-prob"], "o-", label="temperature only")
ax.plot(stats["distinct (top_p=0.9)"], stats["avg log-prob (top_p=0.9)"], "s--", label="temperature + top_p=0.9")
for _, r in stats.iterrows():
    ax.annotate(f"T={r['T']}", (r["distinct bigrams"], r["avg log-prob"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("diversity: distinct word pairs over 300 samples"); ax.set_ylabel("quality: avg log-prob per word")
ax.set_title("The diversity-quality trade-off; truncation limits the damage at high T")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
assert stats["distinct bigrams"].is_monotonic_increasing and stats["avg log-prob"].is_monotonic_decreasing
assert stats["avg log-prob (top_p=0.9)"].iloc[-1] > stats["avg log-prob"].iloc[-1]
print("sample at T=1.8:", " ".join(generate_lm("the", n=12, temperature=1.8, rng=np.random.default_rng(1))))

Higher temperature buys diversity at the price of plausibility; adding top-p cuts the implausible tail, so at high temperature you keep much of the diversity with less nonsense. This is exactly the knob you turn for "give me 5 different taglines" vs "extract the invoice number".

## 8. The same knobs through the API: `temperature`, `top_p`, `n`, `seed` and logprobs

`logprobs=True` returns the log-probability of each generated token; `top_logprobs=5` also returns the 5 most likely alternatives at each position.

> **Offline note:** the stand-in returns deterministic canned text, all `n` choices are identical, and its **logprobs are synthetic** (hash-based numbers with the right shape). With an API key or Ollama the same code shows real probabilities: typically near 0 (p≈1) for forced tokens like punctuation and much lower for content choices.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Summarise in one sentence.\n\nThe cat sat on the mat because it was warm. "
                                          "The dog watched from the sofa and did not move all afternoon."}],
    temperature=0.7, top_p=1.0, n=3, seed=42, max_tokens=40,
    logprobs=True, top_logprobs=5,
)
for ch in resp.choices:
    print(f"choice {ch.index}: {ch.message.content!r} (finish_reason={ch.finish_reason})")
print("system_fingerprint:", resp.system_fingerprint, "| usage:", resp.usage.prompt_tokens, "in,", resp.usage.completion_tokens, "out")

In [ ]:
lp = resp.choices[0].logprobs.content
tok_text = [t.token for t in lp]
tok_prob = np.exp([t.logprob for t in lp])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2, 1]})
axes[0].bar(range(len(tok_prob)), tok_prob, color=plt.cm.RdYlGn(tok_prob))
axes[0].set_xticks(range(len(tok_text)), [t.strip() or "␣" for t in tok_text], rotation=60, fontsize=8)
axes[0].set_ylabel("p(token)"); axes[0].set_ylim(0, 1)
axes[0].set_title(f"Per-token probability from logprobs ({PROVIDER})")
alts = lp[0].top_logprobs
axes[1].barh([a.token for a in alts][::-1], np.exp([a.logprob for a in alts])[::-1], color="tab:blue")
axes[1].set_xlabel("probability"); axes[1].set_title("top-5 alternatives for token 1")
plt.tight_layout(); plt.show()

seq_logprob = sum(t.logprob for t in lp)
print(f"sequence log-prob {seq_logprob:.2f} | mean per token {seq_logprob / len(lp):.2f} | "
      f"perplexity {math.exp(-seq_logprob / len(lp)):.2f} | least confident token: {tok_text[int(np.argmin(tok_prob))]!r}")
assert all(t.logprob <= 0 for t in lp), "log-probabilities are never positive"

Legitimate uses of logprobs: **classification confidence** (ask for a one-token label and read `top_logprobs` to get P(yes) vs P(no), then threshold or abstain), **ranking** candidates by sequence log-prob, flagging **low-confidence spans** for review, and calibration checks. Not legitimate: treating them as the probability that the *answer is true* - models can be confidently wrong, and RLHF-tuned models are often over-confident.

### Why `temperature=0` is still not perfectly deterministic

Greedy decoding is deterministic *in maths*, not on a GPU server. Floating-point addition is not associative, and the order of additions inside a matrix multiply depends on how requests are batched together on the server. When two logits are nearly tied, a difference in the last bit flips the argmax - and after one different token, the rest of the answer diverges.

In [ ]:
r = np.random.default_rng(7)
terms = (r.normal(size=4096) * r.normal(size=4096)).astype(np.float32)   # products h_i * w_i of a logit
order_a = np.float32(0)
for x in terms:                                  # one reduction order: left to right
    order_a = np.float32(order_a + x)
order_b = terms.reshape(64, 64).sum(axis=1, dtype=np.float32).sum(dtype=np.float32)   # blocked, like a GPU kernel
print(f"same logit, two summation orders: {order_a:.8f} vs {order_b:.8f} (diff {abs(order_a - order_b):.1e})")
assert order_a != order_b

rival = np.float32((np.float64(order_a) + np.float64(order_b)) / 2)      # a competing token sitting in that gap
for name, v in [("order A", order_a), ("order B", order_b)]:
    print(f"{name}: greedy picks {'token 1' if v > rival else 'token 2'}")
assert (order_a > rival) != (order_b > rival), "the greedy choice flips with summation order"

We placed the rival logit in the gap on purpose; with real models such near-ties are rare per token but common over millions of tokens. Other sources: mixture-of-experts routing that depends on the batch, and model updates behind an alias. **What to do (in32):** set `temperature=0` (or low) *and* `seed`, log `system_fingerprint` (it changes when the backend changes), pin a dated model snapshot, and write tests that check *properties* (valid JSON, contains the key facts) rather than exact strings. For true bit-exactness you need your own inference stack with deterministic kernels and fixed batch composition.

### Reasoning models reject `temperature`

GPT-5-family and o-series reasoning models do not accept sampling parameters; you control them with reasoning effort instead. The same request that works with `gpt-4.1-mini` fails:

```python
client.chat.completions.create(model="gpt-5-mini", messages=msgs, temperature=0.2)
# openai.BadRequestError: Error code: 400 - {'error': {'message': "Unsupported value: 'temperature'
#   does not support 0.2 with this model. Only the default (1) value is supported.", ...}}

client.chat.completions.create(model="gpt-5-mini", messages=msgs, reasoning_effort="low",
                               max_completion_tokens=2000)    # reasoning tokens count here too
```

(The offline stand-in does not simulate this rejection. Exact rules change between model releases - check the model page.) So a gateway that routes between models must strip `temperature`/`top_p`/`logprobs` for reasoning models.

## 9. Recommended settings

| Task | temperature | top_p | Other | Why |
|---|---|---|---|---|
| Extraction, classification, JSON | 0 - 0.2 | 1 | structured outputs / JSON schema, `seed` | one right answer; stability matters |
| RAG answers, support bots | 0.1 - 0.4 | 1 | `max_tokens` cap | faithful to context, a little fluency |
| Code generation | 0 - 0.3 (0.6-0.8 when sampling k candidates for pass@k) | 0.95 | tests pick the winner | correctness first; diversity only if you verify |
| Chat, explanations | 0.5 - 0.8 | 1 | mild frequency penalty 0-0.3 | natural, not robotic |
| Brainstorming, creative writing | 0.8 - 1.2 | 0.9 - 0.95 | presence penalty 0.3-0.6, `n` > 1 | diversity is the goal |
| Reasoning models (o-series, GPT-5) | not allowed | not allowed | `reasoning_effort` | sampling is fixed by the provider |

## Try it yourself

**1.** Implement `nucleus_size(logits, top_p, temperature)` and show how the number of tokens kept by top_p=0.9 grows with temperature for our toy logits.

In [ ]:
# Solution — try it yourself first, then run this
def nucleus_size(logits, top_p, temperature):
    return int(np.count_nonzero(top_p_filter(softmax(logits / temperature), top_p)))

sizes = {t: nucleus_size(LOGITS, 0.9, t) for t in [0.2, 0.5, 0.7, 1.0, 1.5, 2.0]}
print("tokens in the 0.9 nucleus by temperature:", sizes)
assert list(sizes.values()) == sorted(sizes.values())

**2.** Classification with logprobs: ask the API for a one-word sentiment label with `max_tokens=1, logprobs=True, top_logprobs=3`, and turn the alternatives into a probability distribution over labels.

In [ ]:
# Solution — try it yourself first, then run this
r1 = client.chat.completions.create(
    model=MODEL, max_tokens=1, temperature=0, logprobs=True, top_logprobs=3,
    messages=[{"role": "user", "content": "Classify the sentiment as one of: positive, negative, neutral.\n"
                                          "Text: The delivery was late and the box was broken."}])
first = r1.choices[0].logprobs.content[0]
dist = {a.token.strip(): round(math.exp(a.logprob), 3) for a in first.top_logprobs}
print("answer:", r1.choices[0].message.content, "| alternatives:", dist)
print("(offline the alternatives are synthetic; a real model puts most mass on 'negative')")

**3.** Beam search width: run `beam_search` with beam=1 and beam=3. Show that beam=1 is exactly greedy.

In [ ]:
# Solution — try it yourself first, then run this
for bw in [1, 2, 3]:
    seq, lpv = beam_search(beam=bw)
    print(f"beam={bw}: {' '.join(seq):14s} P={math.exp(lpv):.2f}")
assert beam_search(beam=1)[0] == greedy_tree()[0], "a beam of width 1 is greedy decoding"

## Say it in an interview

- **Temperature:** "Divide the logits by T before softmax. T<1 sharpens, T>1 flattens, T->0 is argmax. It changes confidence, never the ranking."
- **Top-k vs top-p:** top-k keeps a fixed number of tokens; top-p keeps the smallest set covering p of the mass, so it adapts to confidence - prefer top-p (or min-p). Tune temperature *or* top_p, not both.
- **Order in a sampler:** penalties -> temperature -> top-k -> top-p/min-p -> renormalise -> sample.
- **Penalties:** frequency = proportional to count, presence = flat once seen (OpenAI, additive on logits); HF repetition_penalty divides positive logits. They fix loops but hurt code and JSON if too strong.
- **Beam search:** best for one-right-answer seq2seq (translation, ASR); bland for chat, so chat APIs sample.
- **Determinism:** temperature 0 + seed is best-effort; floating-point order and batching can flip near-ties; log `system_fingerprint`, pin snapshots, test properties not strings.
- **Logprobs:** great for label confidence, ranking and abstention; not a probability of truth.
- **Traps:** reasoning models reject temperature; `n=5` costs 5x the output tokens; high temperature breaks JSON (use structured outputs); `max_tokens` truncation shows up as `finish_reason="length"`.

## Next

- [06 · KV cache and inference speed](06_kv_cache_and_inference_speed.ipynb) - make each decoding step cheap.
- [Prompt engineering](../08_llm_apps/02_prompt_engineering.ipynb) · [Structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb) - the other half of controlling outputs.
- [Tokenizers, models and generate](../10_huggingface/02_tokenizers_models_and_generate.ipynb) - the same knobs in `model.generate()`.
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 5 "Generation", chapter 19 "Decoding controls") · [interview bank](../../ai_interview_prep/index.html)